In [ ]:
# Only for test set

import pandas as pd

# Path to your CSV
csv_path = "gen_vs_gt.csv"   # change if needed

# Load
df = pd.read_csv(csv_path)

# Check expected columns
expected_cols = {"Case ID", "Generated Reports", "Ground Truths"}
missing = expected_cols - set(df.columns)
if missing:
    raise ValueError(f"Missing columns in CSV: {missing}")

def extract_organ(gt_text):
    """
    Assumes organ name is the first part of the Ground Truth,
    before the first comma.
    """
    if not isinstance(gt_text, str):
        return "UNKNOWN"
    return gt_text.split(",", 1)[0].strip()

def count_words(text):
    """
    Simple word count: split on whitespace.
    """
    if not isinstance(text, str):
        return 0
    return len(text.split())

# Add helper columns
df["Organ"] = df["Ground Truths"].apply(extract_organ)
df["gt_word_count"] = df["Ground Truths"].apply(count_words)
df["gen_word_count"] = df["Generated Reports"].apply(count_words)

# Group by organ and compute stats
summary = (
    df
    .groupby("Organ")
    .agg(
        n_reports=("Case ID", "count"),
        avg_gt_words=("gt_word_count", "mean"),
        avg_gen_words=("gen_word_count", "mean"),
    )
    .reset_index()
    .sort_values("n_reports", ascending=False)
)

# Print nicely
pd.set_option("display.precision", 2)
print(summary)

# If you want to save it:
summary.to_csv("organ_report_lengths.csv", index=False)
print("\nSaved organ-level stats to organ_report_lengths.csv")

             Organ  n_reports  avg_gt_words  avg_gen_words
0           Breast        245         14.81          14.53
3         Prostate        227         13.86          13.79
5          Stomach        160          7.82           7.53
6  Urinary bladder        101         18.57          17.57
2             Lung         93          5.61           5.54
1            Colon         82          6.82           7.01
7   Uterine cervix         70         10.59          10.04
4           Rectum         24          6.42           6.25

Saved organ-level stats to organ_report_lengths.csv


In [ ]:
#For entire dataset

import json
import pandas as pd

# --------- Config ---------
json_path = "train_val_test.json"

# --------- Load JSON ---------
with open(json_path, "r", encoding="utf-8") as f:
    data = json.load(f)

# data is expected to be a dict with keys "train", "val", "test"
rows = []
for split_name in ["train", "val", "test"]:
    if split_name not in data:
        continue
    for item in data[split_name]:
        rows.append({
            "id": item.get("id"),
            "report": item.get("report", ""),
            "split": split_name
        })

df = pd.DataFrame(rows)

# --------- Helper functions ---------
def extract_organ(report_text):
    """Organ assumed to be first part of report before the first comma."""
    if not isinstance(report_text, str):
        return "UNKNOWN"
    return report_text.split(",", 1)[0].strip()

def count_words(report_text):
    if not isinstance(report_text, str):
        return 0
    # optionally normalize whitespace / newlines
    text = report_text.replace("\n", " ")
    return len(text.split())

# --------- Compute organ & length ---------
df["Organ"] = df["report"].apply(extract_organ)
df["word_count"] = df["report"].apply(count_words)

# --------- Group by split and organ ---------
summary = (
    df
    .groupby(["split", "Organ"])
    .agg(
        n_reports=("id", "count"),
        avg_words=("word_count", "mean")
    )
    .reset_index()
    .sort_values(["split", "n_reports"], ascending=[True, False])
)

pd.set_option("display.precision", 2)
print(summary)

# Optionally save for later analysis
summary.to_csv("organ_report_lengths_by_split.csv", index=False)
print("\nSaved to organ_report_lengths_by_split.csv")

In [ ]:
# Example

def count_words(report_text):
    if not isinstance(report_text, str):
        return 0
    # optionally normalize whitespace / newlines
    text = report_text.replace("\n", " ")
    return len(text.split())

text = "Breast, sono-guided core biopsy;\n  1. Papillary neoplasm\n  2. Usual ductal hyperplasia\n  3. Microcalcification"

le=count_words(text)
print(le)

TypeError: 'int' object is not callable